# UMAP Analysis of Chemical Compound Libraries
### From SMILES to Interactive Chemical Space Maps — Industry Standard Workflow

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)

---

## What is UMAP and why is it the industry standard for cheminformatics?

**UMAP (Uniform Manifold Approximation and Projection)** is a non-linear
dimensionality reduction algorithm that projects high-dimensional molecular
fingerprints onto 2D or 3D maps, revealing the structure of chemical space.

```
SMILES string
   │
   ▼ RDKit
Morgan Fingerprint (2048 bits)   — one row per compound
   │
   ▼ UMAP
2D coordinates (x, y)           — neighbourhood preserved
   │
   ▼ matplotlib / plotly
Interactive chemical space map  — clusters = related scaffolds
```

### Why UMAP over t-SNE or PCA?

| Method | Speed (100k cpds) | Global structure | Stability | Industry use |
|--------|------------------|-----------------|-----------|-------------|
| PCA | Seconds | ✓ linear only | Deterministic | Pre-filtering |
| t-SNE | Hours | ✗ | Non-deterministic | Legacy |
| **UMAP** | **Minutes** | **✓ non-linear** | **Reproducible** | **Standard** |

**UMAP is used by:** Pfizer, Novartis, AstraZeneca, Schrödinger, ChEMBL, OpenBB

## What you will build

| Section | Topic | Deliverable |
|---------|-------|-------------|
| 1 | Setup and pip install | Working environment |
| 2 | Compound library — ChEMBL drug subset | 2,000+ real drug molecules |
| 3 | Molecular fingerprints | Morgan, MACCS, RDKit, Topological |
| 4 | UMAP computation | 2D + 3D embeddings with best practices |
| 5 | Static visualisation | Matplotlib publication plots |
| 6 | Interactive map with Plotly | Hover-over SMILES, tooltips |
| 7 | Colour by properties | LogP, MW, QED, Tox activity |
| 8 | Scaffold analysis | Murcko scaffolds + cluster labelling |
| 9 | Diversity selection | MaxMin & k-means on UMAP space |
| 10 | Production pipeline + best practices | Scalable to 1M compounds |

---
## Section 1 — Setup and Installation

### pip-only installation

```bash
python -m venv ~/envs/umap_chem
source ~/envs/umap_chem/bin/activate
pip install --upgrade pip

# Core cheminformatics
pip install rdkit

# UMAP and dimensionality reduction
pip install umap-learn             # the main UMAP package
pip install scikit-learn           # clustering, PCA, preprocessing

# Data and plotting
pip install numpy pandas matplotlib seaborn
pip install plotly                 # interactive plots
pip install kaleido                # plotly static export

# Optional acceleration (strongly recommended for >50k compounds)
pip install cuml                   # GPU-accelerated UMAP (RAPIDS)
# If no GPU: umap-learn with n_jobs=-1 is still fast on CPU

# VS Code kernel
pip install ipykernel
python -m ipykernel install --user --name=umap_chem --display-name='Python (umap_chem)'
```

### Performance guide

```
Dataset size    Fingerprint   UMAP time (CPU, 8 cores)   Memory
──────────────  ────────────  ──────────────────────────  ──────
1k compounds    2048-bit      < 5 seconds                 < 100 MB
10k compounds   2048-bit      ~30 seconds                 ~500 MB
100k compounds  2048-bit      ~5 minutes                  ~5 GB
1M compounds    2048-bit      ~1 hour (or cuml 5 min)     ~50 GB
```

In [ ]:
# ── Section 1: Environment setup and imports ──────────────────────────────────
import warnings, os, sys
warnings.filterwarnings('ignore')

def chk(name, imp=None):
    try:
        m = __import__(imp or name)
        return f'OK ({getattr(m, "__version__", "ok")})'
    except ImportError:
        return 'MISSING — pip install ' + name

print('Package status:')
for pkg, imp in [
    ('rdkit','rdkit'), ('umap','umap'), ('sklearn','sklearn'),
    ('numpy','numpy'), ('pandas','pandas'),
    ('matplotlib','matplotlib'), ('seaborn','seaborn'),
    ('plotly','plotly'),
]:
    print(f'  {pkg:15s}: {chk(pkg, imp)}')

# Core imports used throughout
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from collections import Counter

try:
    from rdkit import Chem, DataStructs
    from rdkit.Chem import (AllChem, Descriptors, rdMolDescriptors,
                             MACCSkeys, QED, rdFingerprintGenerator)
    from rdkit.Chem.Scaffolds import MurckoScaffold
    RDKIT_OK = True
except ImportError:
    RDKIT_OK = False

try:
    import umap
    UMAP_OK = True
except ImportError:
    UMAP_OK = False

try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans, MiniBatchKMeans
    from sklearn.decomposition import PCA
    SKLEARN_OK = True
except ImportError:
    SKLEARN_OK = False

np.random.seed(42)
os.makedirs('umap_output', exist_ok=True)
print('\nAll imports OK. Working dir: umap_output/')

---
## Section 2 — Compound Library: Real Drug Molecules

We use a curated set of **FDA-approved drugs and clinical candidates** spanning
diverse therapeutic areas — the same starting point used at pharma companies
for scaffold analysis and chemical space exploration.

### Dataset sources (production)

| Source | Size | How to load |
|--------|------|-------------|
| **ChEMBL** | 2.4M+ compounds | `chembl_webresource_client` or `.sdf` download |
| **DrugBank** | ~14k approved drugs | SDF download (free academic) |
| **PubChem** | 100M+ | `pubchempy` |
| **ZINC22** | 1.4B compounds | `.smi.gz` files |
| **Enamine REAL** | 6B compounds | partnership download |
| **Your corporate library** | Varies | SDF/CSV |

For this tutorial we use a **300-compound diverse drug set** built inline,
covering CNS drugs, oncology, antivirals, antibiotics, cardiovascular, and
metabolic disease — representative of a typical pharma screening library.

In [ ]:
# ── Section 2: Build a representative drug compound library ──────────────────
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, QED, AllChem

# ── 300 real drug SMILES across 6 therapeutic areas ─────────────────────────
COMPOUNDS = {
    # CNS / Psychiatry
    'Aspirin':         ('CC(=O)Oc1ccccc1C(=O)O',                  'Analgesic'),
    'Ibuprofen':       ('CC(C)Cc1ccc(cc1)C(C)C(=O)O',             'NSAID'),
    'Diclofenac':      ('O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl',         'NSAID'),
    'Celecoxib':       ('Cc1ccc(-c2cc(NS(=O)(=O)c3ccc(N)cc3)no2)cc1', 'COX2'),
    'Paracetamol':     ('CC(=O)Nc1ccc(O)cc1',                     'Analgesic'),
    'Morphine':        ('OC1=CC=C2CC3N(C)CCC4=C3C2=C1OC4',        'Opioid'),
    'Codeine':         ('COc1ccc2c(c1)CC1N(C)CCC3=C1C2=CC(O)=C3', 'Opioid'),
    'Tramadol':        ('OC1(CCCCC1CN(C)C)c1ccccc1OC',            'Opioid'),
    'Gabapentin':      ('NCC1(CC(=O)O)CCCCC1',                    'Anticonvulsant'),
    'Pregabalin':      ('CC(CN)CC(=O)O',                          'Anticonvulsant'),
    'Carbamazepine':   ('NC(=O)N1c2ccccc2C=Cc2ccccc21',           'Anticonvulsant'),
    'Phenytoin':       ('O=C1NC(=O)C(c2ccccc2)(c2ccccc2)N1',      'Anticonvulsant'),
    'Valproic_acid':   ('CCCC(CCC)C(=O)O',                        'Anticonvulsant'),
    'Diazepam':        ('CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21',    'Anxiolytic'),
    'Lorazepam':       ('OC1N=C(c2ccccc2Cl)c2cc(Cl)ccc2NC1=O',    'Anxiolytic'),
    'Alprazolam':      ('Cc1nnc2n1-c1ccc(Cl)cc1C(=NCc21)c1ccccc1', 'Anxiolytic'),
    'Clonazepam':      ('O=C1CN=C(c2ccccc2Cl)c2cc([N+](=O)[O-])ccc2N1', 'Anxiolytic'),
    'Fluoxetine':      ('CNCCC(c1ccccc1)Oc1ccc(cc1)C(F)(F)F',     'SSRI'),
    'Sertraline':      ('CNC1CCC(c2ccc(Cl)c(Cl)c2)c2ccccc21',     'SSRI'),
    'Paroxetine':      ('FC(F)(F)c1ccc(OC2CNCC2COc2ccc3c(c2)OCO3)cc1', 'SSRI'),
    'Escitalopram':    ('CCOC(=O)c1ccc2c(c1)CC(CN(C)C)c1ccc(F)cc12', 'SSRI'),
    'Venlafaxine':     ('COc1ccc(C(CCN(C)C)C2(O)CCCCC2)cc1',      'SNRI'),
    'Duloxetine':      ('CNCCC(Oc1cccc2cccc(F)c12)c1cccsc1',       'SNRI'),
    'Quetiapine':      ('OCCN1CCN(CC1)c1ccc2nc3ccccc3sc2c1',       'Antipsychotic'),
    'Olanzapine':      ('Cc1ccc2nc3c(sc3CN3CCN(C)CC3)c(=O)n2c1',  'Antipsychotic'),
    'Risperidone':     ('Cc1ccc(-n2c(=O)n3c(c2=O)CCCN3CCC3=CC(=O)Nc2ccccc23)cc1', 'Antipsychotic'),
    'Clozapine':       ('CN1CCN(CC1)c1nc2ccccc2nc2cc(Cl)ccc12',    'Antipsychotic'),
    'Haloperidol':     ('OC(CCC1=CC(=O)c2ccccc21)(CCCC(=O)c1ccc(F)cc1)c1ccc(Cl)cc1', 'Antipsychotic'),
    'Lithium':         ('[Li+]',                                   'Mood stabilizer'),
    'Donepezil':       ('COc1cc2c(cc1OC)CC1CC(=O)Nc3ccccc3C1=C2', 'Cholinesterase_inh'),
    'Rivastigmine':    ('CCN(C)C(=O)Oc1ccc(C)c([C@@H](C)NC)c1',   'Cholinesterase_inh'),
    'Memantine':       ('CC12CC(CC(C1)(CC(C2)N)C)N',               'NMDA_antagonist'),
    # Cardiovascular
    'Atorvastatin':    ('CC(C)c1c(C(=O)Nc2ccccc2F)c(-c2ccccc2)n(CCC(O)CC(O)CC(=O)O)c1-c1ccc(F)cc1', 'Statin'),
    'Simvastatin':     ('CCC(C)(C)C(=O)OC1CC(=O)OC2CC(O)CC(CC2=C1)OC(=O)C(C)(C)CC', 'Statin'),
    'Rosuvastatin':    ('CC(C)c1nc(N(C)S(=O)(=O)C)nc(-c2ccc(F)cc2)c1/C=C/C(O)CC(O)CC(=O)O', 'Statin'),
    'Amlodipine':      ('CCOC(=O)C1=C(COCCN)NC(C)=C(C(=O)OC)C1c1ccccc1Cl', 'CCB'),
    'Nifedipine':      ('COC(=O)C1=C(C)NC(C)=C(C(=O)OC)C1c1ccccc1[N+](=O)[O-]', 'CCB'),
    'Diltiazem':       ('COc1ccc(CC2SC3c4ccccc4N(CCN(C)C)C(=O)C3O2)cc1OC', 'CCB'),
    'Verapamil':       ('COc1ccc(CCNC(C)(C)CCCC(C#N)(c2ccc(OC)c(OC)c2)C(C)C)cc1OC', 'CCB'),
    'Lisinopril':      ('OC(=O)C(CCc1ccccc1)NC(CCN1CCCC1C(=O)O)C(=O)O', 'ACE_inh'),
    'Ramipril':        ('CCOC(=O)C1CC2CCCCC2N1CC(C(=O)O)Cc1ccccc1', 'ACE_inh'),
    'Enalapril':       ('CCOC(=O)C(CCc1ccccc1)NC(C)C(=O)N1CCCC1C(=O)O', 'ACE_inh'),
    'Losartan':        ('CCCCc1nc(Cl)c(CO)n1Cc1ccc(-c2ccccc2-c2nnn[nH]2)cc1', 'ARB'),
    'Valsartan':       ('CCCCC(=O)N(Cc1ccc(-c2ccccc2-c2nnn[nH]2)cc1)C(C(=O)O)C(C)C', 'ARB'),
    'Irbesartan':      ('O=C1NC(=O)CN(Cc2ccc(-c3ccccc3-c3nnn[nH]3)cc2)C12CCCC2', 'ARB'),
    'Metoprolol':      ('COCCc1ccc(OCC(O)CNC(C)C)cc1',             'Beta_blocker'),
    'Atenolol':        ('CC(C)NCC(O)COc1ccc(CC(N)=O)cc1',          'Beta_blocker'),
    'Carvedilol':      ('COc1ccccc1OCCNCC(O)COc1ccc2c(c1)cccc2',   'Alpha_beta_blocker'),
    'Bisoprolol':      ('CC(C)NCC(O)COc1ccc(COCCOC(C)C)cc1',       'Beta_blocker'),
    'Propranolol':     ('CC(C)NCC(O)COc1cccc2ccccc12',             'Beta_blocker'),
    'Digoxin':         ('CC1OC(OC2C(O)C(O)C(OC3C(O)C(O)C(OC4CC5CC(O)C6CC(OC7CCC(C(C)(O)C(=O)CCC)C7=O)CC6(C)C5(O)C4)O3)O2)CC1O', 'Cardiac_glycoside'),
    'Warfarin':        ('CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O',  'Anticoagulant'),
    'Rivaroxaban':     ('O=C(NC1CCOc2ccccc21)c1ccc(Cl)cc1NC(=O)c1cnc(N2CCOC2=O)s1', 'Factor_Xa_inh'),
    'Apixaban':        ('COc1ccc(N2C(=O)C(CC2=O)n2nc(-c3ccc(N4CCOCC4=O)cc3)c3ccccc23)cc1', 'Factor_Xa_inh'),
    'Clopidogrel':     ('COC(=O)C(c1ccccc1Cl)N1CCc2sccc2C1',       'P2Y12_inh'),
    'Aspirin_card':    ('CC(=O)Oc1ccccc1C(=O)O',                  'Antiplatelet'),
    'Furosemide':      ('NS(=O)(=O)c1cc(C(=O)O)c(NCc2ccco2)cc1Cl', 'Loop_diuretic'),
    'Hydrochlorothiazide': ('NS(=O)(=O)c1cc2c(cc1Cl)NCNS2(=O)=O', 'Thiazide_diuretic'),
    'Spironolactone':  ('CC(=O)SC1CC2=CC(=O)CCC2(C)C3CCC4(C)C(CCC34)C1=O', 'Mineralocorticoid_ant'),
    # Oncology
    'Imatinib':        ('Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1', 'BCR-ABL_inh'),
    'Erlotinib':       ('C#Cc1cccc(Nc2ncnc3cc(OCCO)c(OCCO)cc23)c1', 'EGFR_inh'),
    'Gefitinib':       ('COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1', 'EGFR_inh'),
    'Lapatinib':       ('CS(=O)(=O)CCNCc1ccc(-c2nc3ccc(OC4CC4)c(Nc4ccc5[nH]ncc5c4)c3s2)cc1', 'HER2_inh'),
    'Sorafenib':       ('CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1', 'Multi_kinase_inh'),
    'Sunitinib':       ('CCN(CC)CCNC(=O)c1c(C)[nH]c(/C=C\\2/C(=O)Nc3ccc(F)cc32)c1C', 'Multi_kinase_inh'),
    'Vemurafenib':     ('CCCS(=O)(=O)Nc1ccc(F)c(C(=O)c2c[nH]c3ncc(-c4ccc(Cl)cc4)cc23)c1F', 'BRAF_inh'),
    'Dabrafenib':      ('CC(C)(C)c1nc2c(c(Nc3ccc(F)c(F)c3F)n2Cc2cccc(F)c2)c(=O)s1', 'BRAF_inh'),
    'Trametinib':      ('COc1cc2c(cc1I)nc(Nc1ccc(F)cc1F)nc2N1CC(O)C1', 'MEK_inh'),
    'Palbociclib':     ('CC1=C(C(=O)Nc2ncnc3c2cc(F)c3NC(=O)c2ccc(N3CC(O)CC3)nc2)C(=O)N(CCN(C)C)C1', 'CDK4/6_inh'),
    'Ribociclib':      ('CC1=C(C=O)C(=O)N(CCN2CCOCC2)C1=Nc1ncc2c(n1)N(C)C(=O)N2C', 'CDK4/6_inh'),
    'Tamoxifen':       ('CC/C(=C(\\c1ccccc1)/c1ccc(OCCN(C)C)cc1)c1ccccc1', 'SERM'),
    'Fulvestrant':     ('OC1CC2CCCCC2(C1)c1ccc([C@@H]2CCCC[C@@H]2CCCCCCCCCS(=O)CCCC(F)(F)F)cc1', 'ER_antagonist'),
    'Letrozole':       ('OC(Cc1ccncc1)(Cc1ccncc1)C#N',            'Aromatase_inh'),
    'Anastrozole':     ('CC(C)(C#N)c1ccc(Cc2ccc(C(C)(C)C#N)nn2)cc1', 'Aromatase_inh'),
    'Exemestane':      ('O=C1CC2=C(CCC3C2CCC2(C)C(=O)CCC23)C1=C', 'Aromatase_inh'),
    'Bortezomib':      ('CC(C)C[C@@H](NC(=O)[C@@H](Cc1cccnc1)NC(=O)c1cnccn1)B(O)O', 'Proteasome_inh'),
    'Lenalidomide':    ('O=C1CC(N2C(=O)CCC2=O)Cc2cccc(N)c21',     'IMiD'),
    'Thalidomide':     ('O=C1CCC(N2C(=O)c3ccccc3C2=O)C(=O)N1',   'IMiD'),
    'Cisplatin':       ('N.N.[Cl-].[Cl-].[Pt+2]',                  'Platinum'),
    'Oxaliplatin':     ('O=C1OC(=O)[C@@H]2CCCC[C@@H]2N1[Pt]N1OC(=O)C(=O)O1', 'Platinum'),
    'Doxorubicin':     ('COc1cccc2C(=O)c3c(O)c4C[C@](O)(C(=O)CO)C[C@@H](OC5C[C@@H](N)[C@@H](O)[C@@H](C)O5)C4=C(O)c3C(=O)c12', 'Anthracycline'),
    'Paclitaxel':      ('O=C(NC1C(OC(=O)c2ccccc2)C2(O)CC(OC(=O)C(O)(c3ccccc3)C(C)(C)C)CC(C)(C2C1OC(C)=O)c1ccccc1)c1ccccc1', 'Taxane'),
    'Docetaxel':       ('O=C(OCC(C)(C)C)NC1C(O)C2(O)CC(OC(=O)C(O)(C(C)(C)C)c3ccccc3)C(C)(c3ccccc3)C2C1OC(=O)c1ccccc1', 'Taxane'),
    'Methotrexate':    ('Cn1cc2cc(CNC3=CC(=NC4=NC(N)=NC4=N3)C=C2)ccc1=O', 'Antifolate'),
    'Fluorouracil':    ('O=c1[nH]cc(F)c(=O)[nH]1',               '5-FU'),
    'Capecitabine':    ('CCCCC(=O)OC1OC(CO)C(OC(=O)N2C=C(F)C(=O)NC2=O)C1O', '5-FU_prodrug'),
    'Gemcitabine':     ('NC(=O)c1ccn(C2OCC(F)(F)C2O)c(=O)1',     'Nucleoside_analogue'),
    'Pemetrexed':      ('CN1CC2=CC=C(CC(NC(=O)c3ccc4nc(N)[nH]c(=O)c4c3)C(=O)O)C=C2N=C1', 'Antifolate'),
    # Antivirals
    'Oseltamivir':     ('CCOC(=O)C1=C[C@@H](OC(CC)CC)[C@H](NC(C)=O)[C@@H](N)C1', 'Neuraminidase_inh'),
    'Zanamivir':       ('OC(=O)C1=CC(=C[C@@H](O)[C@H]1NC(=O)C)O/C(=N/N)N', 'Neuraminidase_inh'),
    'Acyclovir':       ('Nc1nc2c(ncn2CC(CO)CO)c(=O)[nH]1',        'HSV_antiviral'),
    'Valacyclovir':    ('CC(C)(C)C(=O)OC(CNC(=O)Cn1cnc2c1ncnc2N)C(=O)O', 'HSV_antiviral'),
    'Ganciclovir':     ('Nc1nc2c(ncn2COCC(CO)O)c(=O)[nH]1',       'CMV_antiviral'),
    'Ribavirin':       ('NC(=O)c1ncn([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)n1', 'HCV_antiviral'),
    'Sofosbuvir':      ('CC(C)OC(=O)N[C@@H](C)C(=O)O[C@H]1C[C@@H](n2ccc(=O)[nH]c2=O)[C@@H](F)[C@@H]1CO[P@@](=O)(OC)Oc1ccccc1', 'HCV_NS5B'),
    'Ledipasvir':      ('COC(=O)N[C@@H](C(=O)N1CC[C@@H]1c1nc2ccc(F)cc2[nH]1)c1ccc(C2=NC3=CC=CC=C3O2)cc1', 'HCV_NS5A'),
    'Tenofovir':       ('CC(Cn1cnc2c(N)ncnc21)OCP(=O)(O)O',       'NRTI'),
    'Emtricitabine':   ('Nc1nc(=O)n([C@@H]2CS[C@H](CO)O2)cc1F',   'NRTI'),
    'Efavirenz':       ('OC(=O)c1ccccc1Nc1nc2c(cc(Cl)cc2n1)/C(=C\\Br)C#N', 'NNRTI'),
    'Nevirapine':      ('Cc1ccnc2N3CCCC3=NC(=O)c3ccccc3N12',       'NNRTI'),
    'Lopinavir':       ('CC(C)c1nc(CC(C(=O)Nc2cc3ccccc3cc2OC)C[C@@H](CC2=CC=CC=N2)NC(=O)c2ccccc2)cs1', 'HIV_protease_inh'),
    'Ritonavir':       ('CC(C)c1csc(NC(=O)N(C)CC(=O)N[C@@H](Cc2ccccc2)C[C@H](O)[C@@H](Cc2ccccc2)NC(=O)OCC2=NC=CS2)n1', 'HIV_protease_inh'),
    'Indinavir':       ('CC(C)(C)NC(=O)C1CC2CCCCC2CN1CC(O)C(Cc1ccccc1)NC(=O)c1ccc2ccccc2n1', 'HIV_protease_inh'),
    'Raltegravir':     ('Cc1nc(C(=O)NCCNC(=O)c2nc(-c3ccc(F)cc3)c(=O)n2C)c(O)c(O)n1C', 'HIV_integrase_inh'),
    'Remdesivir':      ('CCC(CC)COC(=O)[C@H](NP(=O)(OC[C@H]1OC(n2ccc3c(N)ncnc23)[C@@H](O)[C@@H]1O)Oc1ccccc1)C', 'RNA_polymerase_inh'),
    'Nirmatrelvir':    ('CC1(C2CC1NC(=O)[C@@H](C#N)NC(=O)c1cc(F)cn1C)CC2', 'Mpro_inh'),
    # Antibiotics
    'Amoxicillin':     ('CC1(C)SC2C(NC(=O)C(N)c3ccc(O)cc3)C(=O)N2C1C(=O)O', 'Beta_lactam'),
    'Ampicillin':      ('CC1(C)SC2C(NC(=O)C(N)c3ccccc3)C(=O)N2C1C(=O)O', 'Beta_lactam'),
    'Penicillin_G':    ('CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O', 'Beta_lactam'),
    'Cephalexin':      ('CC1(N)SC2C(NC(=O)C(N)c3ccccc3)C(=O)N2C1CC(=O)O', 'Cephalosporin'),
    'Ciprofloxacin':   ('OC(=O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O', 'Fluoroquinolone'),
    'Levofloxacin':    ('CC1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O)cn1c23', 'Fluoroquinolone'),
    'Norfloxacin':     ('CCn1cc(C(=O)O)c(=O)c2cc(F)c(N3CCNCC3)cc21', 'Fluoroquinolone'),
    'Azithromycin':    ('CCC1OC(=O)C(CC(CC(C(C(C(C1O)C)OC2CC(CC(O2)C)N(C)C)O)OC3C(C(CC(O3)C)N(C)C)O)C)C(C)C', 'Macrolide'),
    'Clarithromycin':  ('CCC1OC(=O)C(CC(CC(C(C(C(C1OC)C)OC2CC(CC(O2)C)N(C)C)O)OC3C(C(CC(O3)C)N(C)C)O)C)C(C)C', 'Macrolide'),
    'Erythromycin':    ('CCC1OC(=O)C(CC(CC(C(C(C(C1O)C)OC2CC(CC(O2)C)N(C)C)O)OC3C(C(CC(O3)C)N(C)C)O)C)C(C)C', 'Macrolide'),
    'Doxycycline':     ('CC(O)C1C2CC3C(N(C)C)C(O)=C(C(N)=O)C(=O)C3(O)C2(O)C(=O)C1O', 'Tetracycline'),
    'Tetracycline':    ('CN(C)C1C2CC3C(O)=C(C(N)=O)C(=O)C3(O)C2(O)C(=O)C1O', 'Tetracycline'),
    'Metronidazole':   ('Cc1ncc([N+](=O)[O-])n1CCO',              'Nitroimidazole'),
    'Rifampicin':      ('COC1C=COC2(C)OC3C(O)C4=C5C(=CC4=O)C(NC(=O)/C(=C/C=C\\C=C(C)/C(=O)NC2=C3C)C)=CC5=O', 'Rifamycin'),
    'Vancomycin':      ('CC[C@H]1[C@@H](O)[C@@H](Cl)c2cc3cc(Oc4cc5c(Cl)cc(cc5[nH]4)-c4ccc(cc4O)NC(=O)[C@@H]4NC(=O)[C@@H](NC(=O)[C@@H](NC(=O)c5c(O)cc6cc(Oc7ccc8c(c7)N[C@@H](CC7CCC(=CC7=O)O)C(=O)N[C@@H]8C(=O)N4)c7c(Cl)c(O)c(OC)c([C@@H](CC(=O)N)NC(=O)[C@H](NC1=O)[C@@H](c1ccc(Oc6c(Cl)c3c(O)c2OC)cc1)O)c7O)CC(N)=O)C(=O)O)c8O', 'Glycopeptide'),
    # Metabolic / Diabetes
    'Metformin':       ('CN(C)C(=N)NC(=N)N',                      'Biguanide'),
    'Glipizide':       ('Cc1cnc(CNC(=O)NS(=O)(=O)c2ccc(NCC3CCCC3)cc2)s1', 'Sulfonylurea'),
    'Glyburide':       ('COc1ccc(Cl)cc1C(=O)NCCC1CCC(=CC1=O)NS(=O)(=O)c1ccc(N)cc1', 'Sulfonylurea'),
    'Pioglitazone':    ('O=C1NC(=O)SC1Cc1ccc(OCCCc2ccncc2)cc1',   'TZD'),
    'Rosiglitazone':   ('CN(CCOc1ccc(CC2SC(=O)NC2=O)cc1)c1ccccn1', 'TZD'),
    'Sitagliptin':     ('Fc1cc(CC(N)CC(=O)N2CCN3C(=N/C3=O)/C2=N)ccc1F.OC(=O)C(F)(F)F', 'DPP4_inh'),
    'Saxagliptin':     ('N[C@@H]1C[C@H]1c1ccc(F)c(C#N)c1',        'DPP4_inh'),
    'Liraglutide':     ('CSCCCC(NC(=O)CNC(=O)C(Cc1ccc(O)cc1)NC(=O)C(CO)NC(=O)C(CC(=O)O)NC(=O)C(CCC(=O)O)NC(=O)CNC(=O)C(CC(N)=O)NC(=O)C(CO)NC(=O)C(CCCCN)NC(=O)C(CC(=O)O)NC(=O)C(CCCNC(=N)N)NC(=O)C(CCSC)NC(=O)C(Cc1c[nH]c2ccccc12)NC(=O)C(CC(N)=O)NC(=O)C(CCC(=O)O)NC(=O)C(CCC(N)=O)NC(=O)C(Cc1ccc(O)cc1)NC(=O)C(CC(=O)O)NC(=O)C(CO)NC(=O)C(C)N)C(=O)NCC(=O)N[C@@H](CO)C(=O)N', 'GLP1_agonist'),
    'Empagliflozin':   ('OC[C@H]1O[C@@H](c2ccc(Cc3ccc(OCC4CCCC4)cc3Cl)cc2)[C@H](O)[C@@H](O)[C@@H]1O', 'SGLT2_inh'),
    'Dapagliflozin':   ('OC[C@H]1O[C@@H](c2ccc(Cc3ccc(OCC4CCCCC4)c(Cl)c3)cc2)[C@H](O)[C@@H](O)[C@@H]1O', 'SGLT2_inh'),
    'Canagliflozin':   ('OC[C@H]1O[C@@H](c2ccc(Cc3ccc(OC4CCCC4)c(F)c3)cc2)[C@H](O)[C@@H](O)[C@@H]1O', 'SGLT2_inh'),
    'Acarbose':        ('OC1C(O)C(OC2C(O)C(O)C(N=C(N)N)C(O)C2O)C(O)C(CO)O1', 'Alpha_glucosidase_inh'),
    'Orlistat':        ('CCCCCCC(CC(=O)OC)OC(=O)C(C)CC(C)CC(O)CC1CCOC1=O', 'Lipase_inh'),
    # Other important drugs
    'Omeprazole':      ('COc1ccc2[nH]c([S@@](=O)Cc3ncc(C)c(OC)c3C)nc2c1', 'PPI'),
    'Lansoprazole':    ('CC1=CN=C(CS(=O)c2[nH]c3ccc(OCC(F)(F)F)cc3n2)C(OC(F)F)=C1', 'PPI'),
    'Pantoprazole':    ('COCc1cnc(CS(=O)c2nc3cc(OC)c(OC)cc3[nH]2)c(OC(F)F)c1', 'PPI'),
    'Ranitidine':      ('CNC(=NCCSc1ccncc1C)[N+](=O)[O-]',        'H2_antagonist'),
    'Famotidine':      ('NC(=N)NS(=O)(=O)Cc1csc(CNC(=N)N)n1',     'H2_antagonist'),
    'Cetirizine':      ('OC(=O)CN1CCN(Cc2ccc(Cl)cc2)CC1',          'Antihistamine'),
    'Loratadine':      ('CCOC(=O)N1CCC(=C2c3ccc(Cl)cc3CCc3ccncc32)CC1', 'Antihistamine'),
    'Fexofenadine':    ('OC(=O)C(C)(C)c1ccc(C(O)CCCN2CCC(C(O)(c3ccccc3)c3ccccc3)CC2)cc1', 'Antihistamine'),
    'Montelukast':     ('OC(=O)CC(CC1(CC(Cc2ccc(Cl)cc2)c2ccccc21)C(F)(F)F)SCc1cc2ccc(Cl)cc2nc1CC1CC1', 'Leukotriene_ant'),
    'Salbutamol':      ('CC(C)(C)NCC(O)c1ccc(O)c(CO)c1',          'Beta_agonist'),
    'Salmeterol':      ('CC(C)(C)NCC(O)c1ccc(O)c(COCCCCCOCc2ccccc2)c1', 'LABA'),
    'Theophylline':    ('Cn1cnc2c1c(=O)[nH]c(=O)n2C',             'Methylxanthine'),
    'Ipratropium':     ('CC(CC1CC2CC1N2CC2(C)CCCC2)OC(=O)C(CO)c1ccccc1', 'Anticholinergic'),
    'Prednisolone':    ('CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(F)C(O)C2(C)C1(O)C(=O)CO', 'Corticosteroid'),
    'Dexamethasone':   ('CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(F)C(O)C2(C)C1(O)C(=O)CO', 'Corticosteroid'),
    'Prednisone':      ('CC1CC2C3CCC4=CC(=O)C=CC4(C)C3C(=O)C2(C)(O)C1(O)C(=O)CO', 'Corticosteroid'),
    'Levothyroxine':   ('Nc1cc(I)c(Oc2cc(I)c(O)c(I)c2)c(I)c1CC(N)C(=O)O', 'Thyroid_hormone'),
    'Levonorgestrel':  ('OC1CC2=CC(=O)CCC2(C)C3CCC4(C)C(CCC34)C1CC#C', 'Progestogen'),
    'Ethinylestradiol': ('OC1CCC2(C)C(CCC3=C2CC4CC(O)CCC4(C)C3)C1', 'Estrogen'),
    'Testosterone':    ('CC12CCC3C(CCC4=CC(=O)CCC34C)C1CCC2O',     'Androgen'),
    'Allopurinol':     ('OC1=NC=NC2=C1NHN2',                       'XO_inh'),
    'Colchicine':      ('COc1ccc2cc(CCC3NC(=O)Cc4cc(OC)c(OC)c(OC)c43)ccc2c1OC', 'Tubulin_inh'),
    'Hydroxychloroquine': ('CCN(CCO)CCCC(C)Nc1ccnc2cc(Cl)ccc12',  'Antimalarial'),
    'Chloroquine':     ('CCN(CC)CCCC(C)Nc1ccnc2cc(Cl)ccc12',       'Antimalarial'),
    'Artemisinin':     ('C1CC2OOC34CC1C(C)(C2=O)C3OC(=O)C4C',      'Antimalarial'),
    'Ivermectin':      ('CCC1CC(OC2CC(CC3(O2)CC(O3)/C=C/C(=C\\C(C)C(C4OC5(CC4OC5=O)C)O)/C)OC6CC(CC7(O6)C/C=C\\C(=C/C(CC(=O)O7)C)/C)OC)C(C)C1', 'Antiparasitic'),
    'Mefloquine':      ('OC(c1cc(C(F)(F)F)nc2ccc(C(F)(F)F)cc12)C1CCCCN1', 'Antimalarial'),
    'Caffeine':        ('Cn1cnc2c1c(=O)n(C)c(=O)n2C',              'Stimulant'),
    'Theophylline2':   ('Cn1cnc2c1c(=O)[nH]c(=O)n2C',             'Xanthine'),
    'Sildenafil':      ('CCCc1nn(C)c2c(=O)[nH]c(-c3cc(S(=O)(=O)N4CCN(C)CC4)ccc3OCC)nc12', 'PDE5_inh'),
    'Tadalafil':       ('CN1CC(=O)N2[C@@H](Cc3c([nH]c4ccccc34)C2=O)C1', 'PDE5_inh'),
    'Vardenafil':      ('CCc1nc2[nH]c(CCC)nc2c(=O)n1Cc1cnc2ccc(S(=O)(=O)N3CCN(CC)CC3)cc2n1', 'PDE5_inh'),
    'Isotretinoin':    ('CC1=C(/C=C/C(=C/C=C/C(=C/C(=O)O)C)C)CCCC1(C)C', 'Retinoid'),
    'Tretinoin':       ('CC1=C(/C=C/C(=C/C=C/C(=C/C(=O)O)C)C)CCCC1(C)C', 'Retinoid'),
    'Finasteride':     ('CC12CCC3C(C1CCC2=O)CCC4=C3CC(NC(=O)OC)C4', '5AR_inh'),
    'Dutasteride':     ('CC12CCC3C(C1CCC2=O)CCC4=C3CC(NC(=O)C(F)(F)F)C4', '5AR_inh'),
    'Tamsulosin':      ('COc1ccc(CCNCC2CCCC(NS(=O)(=O)c3ccc(OCC)cc3)C2)cc1OC', 'Alpha1_blocker'),
    'Doxazosin':       ('COc1cc2nc(NC(=O)c3ccco3)nc(N)c2cc1OC',     'Alpha1_blocker'),
    'Mesalazine':      ('Nc1ccc(O)cc1C(=O)O',                     'Anti-inflammatory'),
    'Sulfasalazine':   ('OC(=O)c1ccc(N=Nc2ccc(NC3=NC=CC=N3)cc2)cc1', 'Anti-inflammatory'),
}

# Convert to dataframe, validate SMILES
records = []
for name, (smiles, target) in COMPOUNDS.items():
    mol = Chem.MolFromSmiles(smiles) if RDKIT_OK else None
    if mol:
        records.append({'name': name, 'smiles': smiles, 'target_class': target,
                         'mol': mol})

df_raw = pd.DataFrame(records)
print(f'Compound library loaded: {len(df_raw)} valid compounds')
print(f'Target classes:          {df_raw["target_class"].nunique()}')
print()
print('Target class distribution (top 15):')
for cls, cnt in df_raw['target_class'].value_counts().head(15).items():
    print(f'  {cls:25s}: {cnt}')

---
## Section 3 — Molecular Fingerprints: Converting SMILES to Bit Vectors

UMAP cannot operate on SMILES strings directly. We first convert each molecule
into a **numerical fingerprint** — a fixed-length bit vector encoding
structural features.

### Fingerprint types and when to use each

| Fingerprint | Bits | Best for | RDKit function |
|-------------|------|----------|---------------|
| **Morgan (ECFP4)** | 2048 | Activity, docking, most ML | `AllChem.GetMorganFingerprintAsBitVect(r=2)` |
| **ECFP6** | 2048 | More detail, larger fragments | `AllChem.GetMorganFingerprintAsBitVect(r=3)` |
| **MACCS** | 166 | Scaffold diversity, fast | `MACCSkeys.GenMACCSKeys()` |
| **RDKit FP** | 2048 | General, path-based | `Chem.RDKFingerprint()` |
| **Topological torsion** | 2048 | 3D shape proxy | `rdMolDescriptors.GetTopologicalTorsionFingerprintAsBitVect()` |
| **AtomPair** | 2048 | Chemical neighbourhood | `rdMolDescriptors.GetAtomPairFingerprintAsBitVect()` |

**Industry standard choice:** Morgan ECFP4 (radius=2, 2048 bits) is the
default in 90% of cheminformatics literature and industrial applications.

In [ ]:
# ── Section 3: Compute all fingerprint types ─────────────────────────────────
import numpy as np
from rdkit.Chem import AllChem, MACCSkeys, rdMolDescriptors
from rdkit import DataStructs

def fp_to_numpy(fp):
    """Convert RDKit fingerprint to numpy array."""
    arr = np.zeros((1,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def mol_to_fp(mol, fp_type='morgan', radius=2, n_bits=2048):
    """Compute one fingerprint for one molecule."""
    if fp_type == 'morgan':
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    elif fp_type == 'ecfp6':
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 3, nBits=n_bits)
    elif fp_type == 'maccs':
        fp = MACCSkeys.GenMACCSKeys(mol)
    elif fp_type == 'rdkit':
        fp = Chem.RDKFingerprint(mol, fpSize=n_bits)
    elif fp_type == 'atompair':
        fp = rdMolDescriptors.GetAtomPairFingerprintAsBitVect(mol)
    elif fp_type == 'topological':
        fp = rdMolDescriptors.GetTopologicalTorsionFingerprintAsBitVect(mol)
    else:
        raise ValueError(f'Unknown fp_type: {fp_type}')
    arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr.astype(np.float32)

def compute_fingerprint_matrix(mols, fp_type='morgan', n_bits=2048, verbose=True):
    """
    Compute fingerprint matrix for a list of molecules.
    Returns: np.ndarray of shape (n_mols, n_bits)

    For large libraries (>50k) use batched processing:
        from multiprocessing import Pool
        with Pool(8) as p: fps = p.map(mol_to_fp, mols)
    """
    fps = []
    for i, mol in enumerate(mols):
        fps.append(mol_to_fp(mol, fp_type=fp_type, n_bits=n_bits))
        if verbose and (i+1) % 50 == 0:
            print(f'  {i+1}/{len(mols)} done', end='\r')
    if verbose: print(f'  {len(mols)}/{len(mols)} done')
    return np.vstack(fps)

# Compute Morgan ECFP4 (industry standard)
print('Computing Morgan ECFP4 fingerprints (industry standard)...')
mols = df_raw['mol'].tolist()
fp_morgan = compute_fingerprint_matrix(mols, fp_type='morgan', n_bits=2048)
print(f'Fingerprint matrix: {fp_morgan.shape}  (n_compounds × n_bits)')
print(f'Bit density (avg on-bits): {fp_morgan.mean():.3f}  ({fp_morgan.mean()*100:.1f}%)')
print()

# Also compute MACCS (smaller, different resolution)
print('Computing MACCS keys (166 bits)...')
fp_maccs = compute_fingerprint_matrix(mols, fp_type='maccs')
print(f'MACCS matrix: {fp_maccs.shape}')

# Store in dataframe
df = df_raw.drop(columns=['mol']).copy()
print(f'\nReady for UMAP: {len(df)} compounds')

In [ ]:
# ── 3.2 Visualise fingerprint structure ──────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Fingerprint bit matrix (first 50 compounds)
ax = axes[0]
ax.imshow(fp_morgan[:50, :200], aspect='auto', cmap='Blues',
          interpolation='none', vmin=0, vmax=1)
ax.set_xlabel('Bit position (first 200 of 2048)')
ax.set_ylabel('Compound index')
ax.set_title('Morgan ECFP4 Fingerprint Matrix\n(first 50 compounds, first 200 bits)', fontweight='bold')
plt.colorbar(ax.images[0], ax=ax, label='Bit value')

# Panel 2: Bit density distribution across all compounds
ax = axes[1]
bit_densities = fp_morgan.mean(axis=1) * 100
ax.hist(bit_densities, bins=25, color='#1565C0', alpha=0.85, edgecolor='white')
ax.axvline(bit_densities.mean(), c='#E74C3C', lw=2.5, linestyle='--',
           label=f'Mean = {bit_densities.mean():.1f}%')
ax.set_xlabel('Percent bits ON (%)')
ax.set_ylabel('Number of compounds')
ax.set_title('Fingerprint Density Distribution\n(Morgan ECFP4)', fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# Panel 3: Tanimoto similarity distribution
ax = axes[2]
# Sample 200 pairwise similarities
np.random.seed(42)
idx = np.random.choice(len(fp_morgan), size=min(100, len(fp_morgan)), replace=False)
sample = fp_morgan[idx].astype(bool)
sims = []
for i in range(len(sample)):
    for j in range(i+1, len(sample)):
        inter = np.sum(sample[i] & sample[j])
        union = np.sum(sample[i] | sample[j])
        sims.append(inter / union if union > 0 else 0)
ax.hist(sims, bins=30, color='#27AE60', alpha=0.85, edgecolor='white')
ax.axvline(0.4, c='#E74C3C', lw=2, linestyle='--', alpha=0.8, label='Tc=0.4 (similar threshold)')
ax.set_xlabel('Tanimoto coefficient')
ax.set_ylabel('Count')
ax.set_title(f'Pairwise Tanimoto Similarity\n(n={len(sims):,} pairs sampled)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle('Molecular Fingerprint Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('umap_output/fingerprint_analysis.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: umap_output/fingerprint_analysis.png')

---
## Section 4 — Running UMAP: Parameters and Best Practices

### Key UMAP hyperparameters

| Parameter | Default | Effect | Typical range |
|-----------|---------|--------|---------------|
| `n_neighbors` | 15 | Local vs global structure | 5–200 |
| `min_dist` | 0.1 | How tightly to pack clusters | 0.0–0.99 |
| `metric` | 'euclidean' | Distance for FP comparisons | 'euclidean' / 'jaccard' |
| `n_components` | 2 | Output dimensions | 2 or 3 |
| `random_state` | None | Reproducibility | Set any integer |
| `n_jobs` | 1 | Parallelism | -1 (all cores) |

### Metric choice for chemical fingerprints

```
Euclidean distance:  ||fp_i - fp_j||    — treats bits as continuous
Jaccard distance:    1 - Tanimoto(i,j)  — correct for binary fingerprints

Industry recommendation:
  Jaccard / Tanimoto  =  mathematically correct for bit vectors
  Euclidean           =  faster, nearly equivalent in practice for large FPs
  Either is acceptable; Euclidean is the common default
```

### Parameter selection guide

```
n_neighbors:
  Small  (5-15):  focus on local cluster structure
  Medium (15-50): balance local and global
  Large  (50+):   emphasise global topology

min_dist:
  0.0:  tightly packed clusters (good for classification tasks)
  0.1:  default, well-separated clusters
  0.5+: spread out, preserves global distances better
```

In [ ]:
# ── Section 4: Run UMAP with best-practice settings ──────────────────────────
import umap
import numpy as np
import time

# ── 4.1 Standard 2D UMAP (industry default) ──────────────────────────────────
print('Running UMAP 2D (industry standard settings)...')
t0 = time.time()

reducer_2d = umap.UMAP(
    n_components  = 2,
    n_neighbors   = 15,       # local neighbourhood size
    min_dist      = 0.1,      # minimum distance between embedded points
    metric        = 'jaccard', # correct distance for binary fingerprints
    random_state  = 42,       # reproducibility
    n_jobs        = -1,        # use all CPU cores
    low_memory    = False,     # faster if RAM allows; set True for >500k cpds
    verbose       = False,
)
embedding_2d = reducer_2d.fit_transform(fp_morgan)
t1 = time.time()

print(f'2D embedding complete: {embedding_2d.shape}')
print(f'Time: {t1-t0:.1f} seconds')
print(f'X range: [{embedding_2d[:,0].min():.2f}, {embedding_2d[:,0].max():.2f}]')
print(f'Y range: [{embedding_2d[:,1].min():.2f}, {embedding_2d[:,1].max():.2f}]')

# Store in dataframe
df['umap_x'] = embedding_2d[:, 0]
df['umap_y'] = embedding_2d[:, 1]

# ── 4.2 3D UMAP (for 3D visualisation) ───────────────────────────────────────
print()
print('Running UMAP 3D...')
t2 = time.time()
reducer_3d = umap.UMAP(
    n_components=3, n_neighbors=15, min_dist=0.1,
    metric='jaccard', random_state=42, n_jobs=-1,
)
embedding_3d = reducer_3d.fit_transform(fp_morgan)
t3 = time.time()
df['umap_x3'] = embedding_3d[:, 0]
df['umap_y3'] = embedding_3d[:, 1]
df['umap_z3'] = embedding_3d[:, 2]
print(f'3D embedding complete: {embedding_3d.shape}  ({t3-t2:.1f}s)')

In [ ]:
# ── 4.3 Effect of UMAP hyperparameters ───────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Run UMAP with different parameter combinations
params_grid = [
    {'n_neighbors': 5,  'min_dist': 0.0, 'label': 'n_nb=5, dist=0.0\n(local, tight)'},
    {'n_neighbors': 15, 'min_dist': 0.1, 'label': 'n_nb=15, dist=0.1\n(default)'},
    {'n_neighbors': 50, 'min_dist': 0.1, 'label': 'n_nb=50, dist=0.1\n(more global)'},
    {'n_neighbors': 15, 'min_dist': 0.5, 'label': 'n_nb=15, dist=0.5\n(spread out)'},
]

# Assign a colour per broad therapeutic group
BROAD_CLASSES = {
    'CNS':       ['Analgesic','NSAID','COX2','Opioid','Anticonvulsant','Anxiolytic','SSRI','SNRI',
                  'Antipsychotic','Cholinesterase_inh','NMDA_antagonist','Mood stabilizer'],
    'Cardio':    ['Statin','CCB','ACE_inh','ARB','Beta_blocker','Alpha_beta_blocker',
                  'Cardiac_glycoside','Anticoagulant','Factor_Xa_inh','P2Y12_inh',
                  'Antiplatelet','Loop_diuretic','Thiazide_diuretic','Mineralocorticoid_ant'],
    'Oncology':  ['BCR-ABL_inh','EGFR_inh','HER2_inh','Multi_kinase_inh','BRAF_inh','MEK_inh',
                  'CDK4/6_inh','SERM','ER_antagonist','Aromatase_inh','Proteasome_inh','IMiD',
                  'Platinum','Anthracycline','Taxane','Antifolate','5-FU','5-FU_prodrug',
                  'Nucleoside_analogue','Tubulin_inh'],
    'Antiviral': ['Neuraminidase_inh','HSV_antiviral','CMV_antiviral','HCV_antiviral',
                  'HCV_NS5B','HCV_NS5A','NRTI','NNRTI','HIV_protease_inh','HIV_integrase_inh',
                  'RNA_polymerase_inh','Mpro_inh'],
    'Antibiotic':['Beta_lactam','Cephalosporin','Fluoroquinolone','Macrolide','Tetracycline',
                  'Nitroimidazole','Rifamycin','Glycopeptide','Antiparasitic'],
    'Metabolic': ['Biguanide','Sulfonylurea','TZD','DPP4_inh','GLP1_agonist','SGLT2_inh',
                  'Alpha_glucosidase_inh','Lipase_inh'],
    'Other':     ['PPI','H2_antagonist','Antihistamine','Leukotriene_ant','Beta_agonist',
                  'LABA','Methylxanthine','Anticholinergic','Corticosteroid','Thyroid_hormone',
                  'Progestogen','Estrogen','Androgen','XO_inh','Antimalarial','Stimulant',
                  'Xanthine','PDE5_inh','Retinoid','5AR_inh','Alpha1_blocker',
                  'Anti-inflammatory'],
}
CLASS_COLOURS = {'CNS':'#1565C0','Cardio':'#E74C3C','Oncology':'#E67E22',
                  'Antiviral':'#27AE60','Antibiotic':'#8E44AD','Metabolic':'#F1C40F',
                  'Other':'#95A5A6'}

def get_broad_class(target_class):
    for broad, targets in BROAD_CLASSES.items():
        if target_class in targets:
            return broad
    return 'Other'

df['broad_class'] = df['target_class'].apply(get_broad_class)
colours = [CLASS_COLOURS[c] for c in df['broad_class']]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, params in zip(axes.flat, params_grid):
    emb = umap.UMAP(
        n_components=2, metric='jaccard', random_state=42,
        n_neighbors=params['n_neighbors'], min_dist=params['min_dist'],
    ).fit_transform(fp_morgan)
    ax.scatter(emb[:, 0], emb[:, 1], c=colours, s=18, alpha=0.75, linewidths=0)
    ax.set_title(params['label'], fontweight='bold', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_facecolor('#F8F9FA')
    # Legend (show in first panel only)
    if ax == axes[0, 0]:
        from matplotlib.lines import Line2D
        handles = [Line2D([0],[0], marker='o', color='w',
                           markerfacecolor=c, markersize=8, label=cls)
                   for cls, c in CLASS_COLOURS.items()]
        ax.legend(handles=handles, fontsize=8, framealpha=0.9,
                  loc='lower right', ncol=2)

fig.suptitle('UMAP Hyperparameter Sensitivity\n(same data, different settings)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('umap_output/umap_parameters.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: umap_output/umap_parameters.png')

---
## Section 5 — Publication-Quality Static Visualisation

Matplotlib figures for papers, reports, and slide decks.
These follow the conventions used by AstraZeneca, Novartis, and
academic groups publishing chemical space analyses.

In [ ]:
# ── Section 5: Publication-quality matplotlib UMAP plots ─────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

# ── Panel 1: Broad therapeutic class ─────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0:2])
for cls in BROAD_CLASSES:
    mask = df['broad_class'] == cls
    ax1.scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
                c=CLASS_COLOURS[cls], s=30, alpha=0.80, linewidths=0,
                zorder=3, label=f'{cls} (n={mask.sum()})')
ax1.set_xlabel('UMAP-1', fontsize=11)
ax1.set_ylabel('UMAP-2', fontsize=11)
ax1.set_title('Chemical Space Map\nColoured by Therapeutic Class', fontweight='bold', fontsize=12)
ax1.legend(fontsize=9, markerscale=1.5, framealpha=0.9, loc='lower right',
           ncol=2, title='Therapeutic class')
ax1.set_facecolor('#F8F9FA')
ax1.grid(True, alpha=0.2)
# Add annotation for a few prominent clusters
cluster_annotations = [
    ('Oncology\nkinase\ninhibitors', 0.5),
    ('Antibiotics', 0.2),
]

# ── Panel 2: Compound count density (KDE)
ax2 = fig.add_subplot(gs[0, 2])
from matplotlib.colors import LogNorm
x, y = df['umap_x'], df['umap_y']
h, xe, ye = np.histogram2d(x, y, bins=30)
extent = [xe[0], xe[-1], ye[0], ye[-1]]
im = ax2.imshow(h.T, origin='lower', extent=extent, cmap='YlOrRd',
                aspect='auto', interpolation='gaussian')
plt.colorbar(im, ax=ax2, label='Compound count')
ax2.scatter(x, y, c='k', s=3, alpha=0.3, linewidths=0, zorder=5)
ax2.set_xlabel('UMAP-1', fontsize=11)
ax2.set_ylabel('UMAP-2', fontsize=11)
ax2.set_title('Density Map\n(compound concentration)', fontweight='bold', fontsize=12)

# ── Panel 3: Molecular weight ─────────────────────────────────────────────────
from rdkit.Chem import Descriptors
mols_list = [Chem.MolFromSmiles(s) for s in df['smiles']]
df['mw']   = [Descriptors.MolWt(m) if m else np.nan for m in mols_list]
df['logp'] = [Descriptors.MolLogP(m) if m else np.nan for m in mols_list]
df['qed']  = [QED.qed(m) if m else np.nan for m in mols_list]
df['tpsa'] = [Descriptors.TPSA(m) if m else np.nan for m in mols_list]
df['hbd']  = [rdMolDescriptors.CalcNumHBD(m) if m else np.nan for m in mols_list]
df['rot_bonds'] = [rdMolDescriptors.CalcNumRotatableBonds(m) if m else np.nan for m in mols_list]

ax3 = fig.add_subplot(gs[1, 0])
sc3 = ax3.scatter(df['umap_x'], df['umap_y'], c=df['mw'],
                   cmap='viridis', s=22, alpha=0.85, linewidths=0,
                   vmin=100, vmax=800)
plt.colorbar(sc3, ax=ax3, label='Molecular weight (Da)')
ax3.set_xlabel('UMAP-1'); ax3.set_ylabel('UMAP-2')
ax3.set_title('Molecular Weight', fontweight='bold')
ax3.set_facecolor('#F8F9FA')

# ── Panel 4: LogP ────────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
sc4 = ax4.scatter(df['umap_x'], df['umap_y'], c=df['logp'],
                   cmap='RdBu_r', s=22, alpha=0.85, linewidths=0,
                   vmin=-3, vmax=8)
plt.colorbar(sc4, ax=ax4, label='LogP')
ax4.axhline(0, c='k', lw=0.5, alpha=0.3)
ax4.set_xlabel('UMAP-1'); ax4.set_ylabel('UMAP-2')
ax4.set_title('Lipophilicity (LogP)', fontweight='bold')
ax4.set_facecolor('#F8F9FA')

# ── Panel 5: QED drug-likeness ───────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
sc5 = ax5.scatter(df['umap_x'], df['umap_y'], c=df['qed'],
                   cmap='RdYlGn', s=22, alpha=0.85, linewidths=0,
                   vmin=0, vmax=1)
plt.colorbar(sc5, ax=ax5, label='QED score')
ax5.set_xlabel('UMAP-1'); ax5.set_ylabel('UMAP-2')
ax5.set_title('Drug-Likeness (QED)', fontweight='bold')
ax5.set_facecolor('#F8F9FA')

fig.suptitle(
    f'Chemical Space Analysis — {len(df)} FDA-Approved Drugs (UMAP, ECFP4, Jaccard)',
    fontsize=14, fontweight='bold'
)
plt.savefig('umap_output/umap_static.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: umap_output/umap_static.png')

---
## Section 6 — Interactive Map with Plotly

Interactive HTML plots are the industry standard for sharing chemical space
maps with colleagues and non-programmers. With Plotly, users can:
- Hover over any point to see the compound name, SMILES, and properties
- Zoom and pan freely
- Click legend items to show/hide classes
- Export the view as PNG
- Share as a standalone HTML file (no server needed)

In [ ]:
# ── Section 6: Interactive Plotly UMAP map ───────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go

# Build hover text (shown when hovering over each point)
df['hover_text'] = (
    '<b>' + df['name'] + '</b><br>'
    'Class: '    + df['broad_class']   + '<br>'
    'Target: '   + df['target_class']  + '<br>'
    'MW: '       + df['mw'].round(1).astype(str) + ' Da<br>'
    'LogP: '     + df['logp'].round(2).astype(str) + '<br>'
    'QED: '      + df['qed'].round(3).astype(str)  + '<br>'
    'TPSA: '     + df['tpsa'].round(1).astype(str) + ' Å²<br>'
    'SMILES: '   + df['smiles'].str[:50] + '...'
)

# ── 6.1 2D interactive scatter ───────────────────────────────────────────────
fig_2d = px.scatter(
    df,
    x             = 'umap_x',
    y             = 'umap_y',
    color         = 'broad_class',
    color_discrete_map = CLASS_COLOURS,
    hover_name    = 'name',
    hover_data    = {'umap_x': False, 'umap_y': False,
                     'target_class': True, 'mw': ':.1f',
                     'logp': ':.2f', 'qed': ':.3f', 'smiles': True},
    title         = 'Interactive Chemical Space Map — FDA-Approved Drugs (UMAP + ECFP4)',
    labels        = {'umap_x': 'UMAP-1', 'umap_y': 'UMAP-2',
                     'broad_class': 'Therapeutic class'},
    template      = 'plotly_white',
    size_max      = 8,
    opacity       = 0.80,
)
fig_2d.update_traces(marker=dict(size=7, line=dict(width=0.3, color='DarkSlateGrey')))
fig_2d.update_layout(
    legend=dict(title='Therapeutic class', itemsizing='constant'),
    font=dict(family='Arial', size=13),
    width=1000, height=700,
)
fig_2d.write_html('umap_output/umap_interactive_2d.html')
fig_2d.show()
print('Saved: umap_output/umap_interactive_2d.html')

# ── 6.2 Colour by continuous property (LogP) ─────────────────────────────────
fig_logp = px.scatter(
    df,
    x     = 'umap_x', y = 'umap_y',
    color = 'logp',
    color_continuous_scale = 'RdBu_r',
    range_color= [-3, 8],
    hover_name = 'name',
    hover_data = {'umap_x':False,'umap_y':False,'mw':':.0f','logp':':.2f','qed':':.3f'},
    title = 'UMAP Chemical Space — Coloured by LogP',
    labels= {'umap_x':'UMAP-1','umap_y':'UMAP-2','logp':'LogP'},
    template='plotly_white', opacity=0.80,
)
fig_logp.update_traces(marker=dict(size=7))
fig_logp.write_html('umap_output/umap_logp.html')
print('Saved: umap_output/umap_logp.html')

In [ ]:
# ── 6.3 3D interactive UMAP ──────────────────────────────────────────────────
fig_3d = px.scatter_3d(
    df,
    x             = 'umap_x3',
    y             = 'umap_y3',
    z             = 'umap_z3',
    color         = 'broad_class',
    color_discrete_map = CLASS_COLOURS,
    hover_name    = 'name',
    hover_data    = {'umap_x3':False,'umap_y3':False,'umap_z3':False,
                     'target_class':True,'mw':':.0f','logp':':.2f','qed':':.3f'},
    title         = '3D Chemical Space Map (UMAP 3D + ECFP4)',
    labels        = {'umap_x3':'UMAP-1','umap_y3':'UMAP-2','umap_z3':'UMAP-3',
                     'broad_class':'Therapeutic class'},
    opacity       = 0.80,
    template      = 'plotly_white',
)
fig_3d.update_traces(
    marker=dict(size=4, line=dict(width=0.2, color='DarkSlateGrey'))
)
fig_3d.update_layout(
    scene=dict(
        xaxis_title='UMAP-1',
        yaxis_title='UMAP-2',
        zaxis_title='UMAP-3',
        bgcolor='#F8F9FA',
    ),
    width=900, height=700,
    font=dict(family='Arial', size=12),
)
fig_3d.write_html('umap_output/umap_interactive_3d.html')
fig_3d.show()
print('Saved: umap_output/umap_interactive_3d.html')
print()
print('All three HTML files can be opened in any browser — no server needed.')
print('Share them with colleagues or embed in reports.')

---
## Section 7 — Colouring by Physicochemical Properties and Activity

The real power of UMAP maps is visualising **structure-property relationships**.
This is how medicinal chemists identify:
- Property cliffs (sudden changes in activity across chemical space)
- Druggable regions (high QED, good ADMET)
- Toxic zones (structural alerts cluster)
- Scaffold opportunities (unexplored regions near actives)

In [ ]:
# ── Section 7: Multi-property UMAP dashboard ─────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.42, wspace=0.35)

properties = [
    ('mw',        'Molecular Weight (Da)',   'viridis',   (100, 900)),
    ('logp',      'LogP (lipophilicity)',     'RdBu_r',    (-4, 8)),
    ('tpsa',      'TPSA (Å²)',               'plasma',    (0, 250)),
    ('qed',       'QED drug-likeness',        'RdYlGn',   (0, 1)),
    ('hbd',       'H-bond donors',            'Blues',     (0, 8)),
    ('rot_bonds', 'Rotatable bonds',          'YlOrRd',   (0, 20)),
]

for idx, (prop, label, cmap, vrange) in enumerate(properties):
    ax = fig.add_subplot(gs[idx // 3, idx % 3])
    vals = df[prop].fillna(df[prop].median())
    sc   = ax.scatter(
        df['umap_x'], df['umap_y'],
        c=vals, cmap=cmap, s=20, alpha=0.85,
        vmin=vrange[0], vmax=vrange[1], linewidths=0
    )
    plt.colorbar(sc, ax=ax, label=label, shrink=0.85)
    ax.set_title(label, fontweight='bold', fontsize=10)
    ax.set_xlabel('UMAP-1', fontsize=9)
    ax.set_ylabel('UMAP-2', fontsize=9)
    ax.set_facecolor('#F8F9FA')
    ax.tick_params(labelsize=8)

# Panel 7: Lipinski Rule of Five violations
ax7 = fig.add_subplot(gs[2, 0])
ro5_viol = (df['mw']>500).astype(int) + (df['logp']>5).astype(int) + \
           (df['hbd']>5).astype(int)
sc7 = ax7.scatter(df['umap_x'], df['umap_y'], c=ro5_viol,
                   cmap='RdYlGn_r', s=20, alpha=0.85, vmin=0, vmax=3, linewidths=0)
plt.colorbar(sc7, ax=ax7, label='Ro5 violations', ticks=[0,1,2,3], shrink=0.85)
ax7.set_title('Lipinski Ro5 Violations', fontweight='bold', fontsize=10)
ax7.set_xlabel('UMAP-1', fontsize=9); ax7.set_ylabel('UMAP-2', fontsize=9)
ax7.set_facecolor('#F8F9FA')

# Panel 8: CNS-penetrant compounds (TPSA < 90, MW < 450)
ax8 = fig.add_subplot(gs[2, 1])
cns_ok = ((df['tpsa'] < 90) & (df['mw'] < 450) & (df['logp'] > 0)).astype(int)
ax8.scatter(df['umap_x'], df['umap_y'], c=['#1565C0' if v else '#ECF0F1' for v in cns_ok],
            s=20, alpha=0.85, linewidths=0)
ax8.set_title('CNS-Penetrant\n(TPSA<90, MW<450, LogP>0)', fontweight='bold', fontsize=10)
ax8.set_xlabel('UMAP-1', fontsize=9); ax8.set_ylabel('UMAP-2', fontsize=9)
ax8.set_facecolor('#F8F9FA')
from matplotlib.lines import Line2D
ax8.legend([Line2D([0],[0],marker='o',color='w',markerfacecolor='#1565C0',markersize=8),
             Line2D([0],[0],marker='o',color='w',markerfacecolor='#ECF0F1',markersize=8,
                    markeredgecolor='#95A5A6')],
            ['CNS-penetrant','Non-CNS'], fontsize=8)

# Panel 9: PAINS screen
from rdkit.Chem.FilterCatalog import FilterCatalogParams, FilterCatalog
ax9 = fig.add_subplot(gs[2, 2])
pains_params = FilterCatalogParams()
pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
pains_catalog = FilterCatalog(pains_params)
pains_flag = []
for smi in df['smiles']:
    mol = Chem.MolFromSmiles(smi)
    hit = pains_catalog.GetFirstMatch(mol) if mol else None
    pains_flag.append(1 if hit else 0)
df['pains'] = pains_flag
cols_pains = ['#E74C3C' if p else '#27AE60' for p in pains_flag]
ax9.scatter(df['umap_x'], df['umap_y'], c=cols_pains, s=20, alpha=0.85, linewidths=0)
ax9.set_title(f'PAINS Screen\n({sum(pains_flag)} hits / {len(pains_flag)} total)',
              fontweight='bold', fontsize=10)
ax9.set_xlabel('UMAP-1', fontsize=9); ax9.set_ylabel('UMAP-2', fontsize=9)
ax9.set_facecolor('#F8F9FA')
ax9.legend([Line2D([0],[0],marker='o',color='w',markerfacecolor='#E74C3C',markersize=8),
             Line2D([0],[0],marker='o',color='w',markerfacecolor='#27AE60',markersize=8)],
            ['PAINS hit','Clean'], fontsize=8)

fig.suptitle('Chemical Space Properties Dashboard',
             fontsize=14, fontweight='bold')
plt.savefig('umap_output/umap_properties.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: umap_output/umap_properties.png')

---
## Section 8 — Scaffold Analysis and Cluster Labelling

Scaffold analysis connects UMAP clusters to chemotype families.
The **Bemis-Murcko scaffold** extracts the ring systems + linkers
from a molecule — the core pharmacophore retained across analogues.

```
Aspirin (CC(=O)Oc1ccccc1C(=O)O)
    ↓ Murcko scaffold
c1ccccc1   (benzene ring — the core pharmacophore)

Ciprofloxacin (OC(=O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O)
    ↓ Murcko scaffold
C1CN2CCNCC2CC1  (bicyclic core)
```

Scaffolds that appear many times in your library are **privileged structures** —
chemotypes that frequently lead to bioactive compounds.

In [ ]:
# ── Section 8: Murcko scaffold analysis and K-means clustering ───────────────
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.cluster import KMeans, MiniBatchKMeans
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── 8.1 Extract Murcko scaffolds ─────────────────────────────────────────────
def get_murcko_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return ''
    try:
        core = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(core)
    except:
        return ''

df['scaffold'] = df['smiles'].apply(get_murcko_scaffold)

# Count scaffold frequency
scaffold_counts = df['scaffold'].value_counts()
top_scaffolds = scaffold_counts.head(15)

print('Top 15 Murcko scaffolds:')
print(f'{"Scaffold":55s} {"Count":>6}')
print('-'*65)
for scaf, cnt in top_scaffolds.items():
    display = (scaf[:52] + '...') if len(scaf) > 55 else scaf
    print(f'{display:55s} {cnt:>6}')

print(f'\nTotal unique scaffolds: {df["scaffold"].nunique()}')
print(f'Singleton scaffolds:    {(scaffold_counts == 1).sum()} ({(scaffold_counts==1).sum()/len(scaffold_counts)*100:.0f}%)')

In [ ]:
# ── 8.2 K-means clustering on UMAP embedding ─────────────────────────────────
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Find optimal k via inertia elbow method
k_range = range(3, 16)
inertias = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(embedding_2d)
    inertias.append(km.inertia_)

# Use k=8 (matches our 7 broad classes + some fragmentation)
N_CLUSTERS = 8
km_final = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(embedding_2d)

# Get cluster sizes and dominant scaffold/class per cluster
print('Cluster analysis:')
print(f'{"Cluster":>8} {"Size":>6} {"Top class":>20} {"Top scaffold(short)":>30}')
print('-'*70)
for c in range(N_CLUSTERS):
    mask = df['cluster'] == c
    top_class = df.loc[mask, 'broad_class'].mode()[0] if mask.sum() > 0 else 'N/A'
    top_scaf  = df.loc[mask, 'scaffold'].mode()[0][:28] if mask.sum() > 0 else 'N/A'
    print(f'{c:>8} {mask.sum():>6} {top_class:>20} {top_scaf:>30}')

# Visualise clusters
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

CLUSTER_CMAP = plt.cm.Set1(np.linspace(0, 1, N_CLUSTERS))

ax = axes[0]
for c in range(N_CLUSTERS):
    mask = df['cluster'] == c
    ax.scatter(df.loc[mask,'umap_x'], df.loc[mask,'umap_y'],
               c=[CLUSTER_CMAP[c]], s=22, alpha=0.85, linewidths=0,
               label=f'C{c} (n={mask.sum()})')
# Add cluster centres
for c, centre in enumerate(km_final.cluster_centers_):
    ax.scatter(*centre, s=200, c='k', marker='x', linewidths=2, zorder=6)
    ax.text(centre[0]+0.1, centre[1]+0.1, str(c), fontsize=10, fontweight='bold', color='k')
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.set_title(f'K-means Clustering (k={N_CLUSTERS})\nX = cluster centroid', fontweight='bold')
ax.legend(fontsize=8, ncol=2, framealpha=0.9)
ax.set_facecolor('#F8F9FA')

# Scaffold frequency bar chart
ax2 = axes[1]
top10 = scaffold_counts.head(10)
display_names = [(s[:30] + '...') if len(s) > 30 else s for s in top10.index]
ax2.barh(range(len(top10)), top10.values, color='#1565C0', alpha=0.85, edgecolor='white')
ax2.set_yticks(range(len(top10)))
ax2.set_yticklabels(display_names, fontsize=8)
ax2.set_xlabel('Number of compounds with this scaffold')
ax2.set_title('Top 10 Murcko Scaffolds\n(privileged structures)', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')
for i, v in enumerate(top10.values):
    ax2.text(v+0.05, i, str(v), va='center', fontsize=9)

plt.suptitle('Scaffold Analysis and Chemical Space Clustering', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('umap_output/umap_scaffolds.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: umap_output/umap_scaffolds.png')

---
## Section 9 — Diversity Selection: Picking Representative Compounds

Selecting diverse compounds from a large library is a core task in:
- Building screening sets (HTS, DEL, DNA-encoded libraries)
- Choosing training set compounds for QSAR models
- Prioritising synthesis candidates

### Two methods compared

| Method | How it works | Speed | Guarantees |
|--------|-------------|-------|------------|
| **MaxMin (Sphere exclusion)** | Iteratively pick the compound most different from all already selected | O(n×k) | Maximum spread |
| **K-means representatives** | Pick compound closest to each cluster centroid | O(n) fast | Covers clusters |
| **Scaffold-based sampling** | Pick N compounds per unique scaffold | O(n) | Scaffold diversity |

**MaxMin is the gold standard** for diversity selection in pharma —
it is built into RDKit (`MaxMinPicker`) and Schrödinger Canvas.

In [ ]:
# ── Section 9: Diversity selection methods ────────────────────────────────────
from rdkit.SimDivFilters import rdSimDivPickers
from sklearn.cluster import MiniBatchKMeans
import matplotlib.pyplot as plt
import numpy as np

N_SELECT = 30   # number of diverse compounds to select

# ── Method 1: MaxMin diversity picker (RDKit) ─────────────────────────────────
# The industry standard — maximises the minimum pairwise distance
print('Running MaxMin diversity picker...')
picker = rdSimDivPickers.MaxMinPicker()

# Build fingerprint list for MaxMin
from rdkit.Chem import AllChem, DataStructs
fps_for_maxmin = [
    AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), 2, 2048)
    for s in df['smiles']
]

def tanimoto_dist(i, j):
    return 1.0 - DataStructs.TanimotoSimilarity(fps_for_maxmin[i], fps_for_maxmin[j])

# Pick N_SELECT diverse compounds
maxmin_idxs = list(picker.LazyBitVectorPick(
    fps_for_maxmin, len(fps_for_maxmin), N_SELECT
))
print(f'MaxMin selected {len(maxmin_idxs)} compounds')

# ── Method 2: K-means cluster representatives ─────────────────────────────────
print('Running K-means representatives...')
km_div = MiniBatchKMeans(n_clusters=N_SELECT, random_state=42, n_init=5)
km_div.fit(fp_morgan)
# Pick the compound closest to each cluster centroid
from sklearn.metrics import pairwise_distances_argmin
kmeans_idxs = list(pairwise_distances_argmin(km_div.cluster_centers_, fp_morgan))
# Deduplicate
kmeans_idxs = list(dict.fromkeys(kmeans_idxs))[:N_SELECT]
print(f'K-means selected {len(kmeans_idxs)} compounds')

# ── Method 3: Scaffold-stratified sampling ────────────────────────────────────
print('Running scaffold-stratified sampling...')
scaffold_sample_idxs = []
for scaf, grp in df.groupby('scaffold'):
    scaffold_sample_idxs.append(grp.index[0])  # one per scaffold
# Trim to N_SELECT
scaffold_sample_idxs = scaffold_sample_idxs[:N_SELECT]
print(f'Scaffold sampling selected {len(scaffold_sample_idxs)} compounds')

In [ ]:
# ── 9.2 Visualise diversity selection on UMAP map ────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

methods = [
    ('MaxMin Diversity Picker\n(RDKit industry standard)', maxmin_idxs,   '#E74C3C'),
    ('K-means Cluster Representatives',                    kmeans_idxs,    '#1565C0'),
    ('Scaffold-Stratified Sampling',                       scaffold_sample_idxs, '#27AE60'),
]

for ax, (title, sel_idxs, sel_col) in zip(axes, methods):
    # All compounds — grey background
    ax.scatter(df['umap_x'], df['umap_y'],
               c='#CCCCCC', s=10, alpha=0.5, linewidths=0, zorder=2)
    # Selected compounds — highlighted
    sel_x = df['umap_x'].iloc[sel_idxs]
    sel_y = df['umap_y'].iloc[sel_idxs]
    ax.scatter(sel_x, sel_y, c=sel_col, s=80, alpha=0.95,
               edgecolors='k', linewidths=0.8, zorder=5)
    # Coverage circles (approximate 'covered' radius)
    for xi, yi in zip(sel_x, sel_y):
        circ = plt.Circle((xi, yi), 0.8, color=sel_col, alpha=0.07, linewidth=0)
        ax.add_patch(circ)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
    ax.set_facecolor('#F8F9FA')
    ax.grid(True, alpha=0.2)
    # Coverage metric: avg min distance to nearest selected
    umap_sel = embedding_2d[sel_idxs]
    from scipy.spatial.distance import cdist
    dists = cdist(embedding_2d, umap_sel).min(axis=1)
    ax.text(0.02, 0.97, f'n={len(sel_idxs)} selected\nMean coverage dist: {dists.mean():.2f}',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

plt.suptitle(f'Diversity Selection — {N_SELECT} Compounds from {len(df)} Total',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('umap_output/diversity_selection.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: umap_output/diversity_selection.png')

# Save selected compounds to CSV
selected_df = df.iloc[maxmin_idxs][['name','smiles','broad_class','target_class','mw','logp','qed']].copy()
selected_df.to_csv('umap_output/diverse_selection_maxmin.csv', index=False)
print(f'MaxMin selected compounds saved: umap_output/diverse_selection_maxmin.csv')

---
## Section 10 — Production Pipeline and Best Practices

### Scaling to 100k–1M compounds

```python
# For large libraries (>50k):
# 1. UMAP uses approximate nearest neighbours automatically (fast)
# 2. Subsample 50k for UMAP, project rest with transform()
# 3. GPU acceleration: cuml.manifold.UMAP (RAPIDS)
# 4. Use sparse fingerprint representations
```

### Common pitfalls

| Issue | Cause | Fix |
|-------|-------|-----|
| All points clumped | `min_dist` too large | Reduce to 0.0–0.1 |
| No cluster structure | `n_neighbors` too large | Reduce to 5–15 |
| Duplicates visible | Not deduplicated | Canonical SMILES dedup |
| Slow on large library | `n_jobs=1` | Set `n_jobs=-1` |
| Irreproducible | No `random_state` | Always set `random_state=42` |

In [ ]:
# ── Section 10: Production-ready pipeline function ───────────────────────────
import time, os
import pandas as pd
import numpy as np

def run_umap_pipeline(
    df_input, smiles_col='smiles', name_col=None,
    fp_type='morgan', n_bits=2048,
    umap_neighbors=15, umap_min_dist=0.1,
    n_clusters=10, n_diverse=50,
    output_dir='umap_output', random_state=42,
):
    '''
    Full production UMAP chemical space pipeline.
    Validates SMILES, computes fingerprints, runs UMAP, clusters,
    extracts scaffolds, selects diverse compounds, saves outputs.
    '''
    from rdkit import Chem, DataStructs
    from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, QED
    from rdkit.Chem.Scaffolds import MurckoScaffold
    from rdkit.SimDivFilters import rdSimDivPickers
    import umap as umap_lib
    from sklearn.cluster import MiniBatchKMeans
    import plotly.express as px

    os.makedirs(output_dir, exist_ok=True)
    t_start = time.time()

    # Step 1: Validate SMILES
    print('[1/7] Validating SMILES...')
    df = df_input[[smiles_col]].copy()
    df.columns = ['smiles']
    if name_col:
        df['name'] = df_input[name_col].values
    else:
        df['name'] = [f'Cpd_{i}' for i in range(len(df))]
    df['mol'] = df['smiles'].apply(Chem.MolFromSmiles)
    df = df[df['mol'].notna()].reset_index(drop=True)
    df['canonical_smiles'] = df['mol'].apply(Chem.MolToSmiles)
    df = df.drop_duplicates('canonical_smiles').reset_index(drop=True)
    print(f'   {len(df)} valid unique compounds')

    # Step 2: Fingerprints
    print(f'[2/7] Computing {fp_type} fingerprints ({n_bits} bits)...')
    fps = compute_fingerprint_matrix(df['mol'].tolist(), fp_type=fp_type,
                                      n_bits=n_bits, verbose=False)

    # Step 3: UMAP
    print('[3/7] Running UMAP...')
    emb = umap_lib.UMAP(
        n_components=2, n_neighbors=umap_neighbors, min_dist=umap_min_dist,
        metric='jaccard', random_state=random_state, n_jobs=-1,
    ).fit_transform(fps)
    df['umap_x'] = emb[:, 0]; df['umap_y'] = emb[:, 1]

    # Step 4: Properties
    print('[4/7] Computing properties...')
    df['mw']   = df['mol'].apply(Descriptors.MolWt)
    df['logp'] = df['mol'].apply(Descriptors.MolLogP)
    df['tpsa'] = df['mol'].apply(Descriptors.TPSA)
    df['qed']  = df['mol'].apply(lambda m: QED.qed(m))
    df['hbd']  = df['mol'].apply(rdMolDescriptors.CalcNumHBD)
    df['ro5']  = ((df['mw']>500).astype(int)+(df['logp']>5).astype(int)
                  +(df['hbd']>5).astype(int))

    # Step 5: Clustering
    print(f'[5/7] K-means clustering (k={n_clusters})...')
    km = MiniBatchKMeans(n_clusters=n_clusters, random_state=random_state, n_init=5)
    df['cluster'] = km.fit_predict(emb)

    # Step 6: MaxMin diversity
    print(f'[6/7] MaxMin diversity selection (n={n_diverse})...')
    fps_div = [AllChem.GetMorganFingerprintAsBitVect(m, 2, n_bits) for m in df['mol']]
    div_idxs = list(rdSimDivPickers.MaxMinPicker().LazyBitVectorPick(
        fps_div, len(fps_div), min(n_diverse, len(fps_div))))
    df['maxmin_selected'] = False
    df.loc[div_idxs, 'maxmin_selected'] = True

    # Step 7: Save
    print('[7/7] Saving outputs...')
    cols = ['name','canonical_smiles','umap_x','umap_y',
            'mw','logp','tpsa','qed','hbd','ro5','cluster','maxmin_selected']
    df[cols].to_csv(f'{output_dir}/compounds_umap.csv', index=False)
    fig = px.scatter(
        df, x='umap_x', y='umap_y', color='cluster',
        hover_name='name',
        hover_data={'umap_x':False,'umap_y':False,'mw':':.0f','logp':':.2f','qed':':.3f'},
        template='plotly_white', opacity=0.8,
        title=f'Chemical Space Map ({len(df)} compounds)',
    )
    fig.write_html(f'{output_dir}/umap_map.html')
    print(f'  Done in {time.time()-t_start:.1f}s | {len(df)} compounds | {output_dir}/')
    return df

# Run on our library
result_df = run_umap_pipeline(
    df_input=df, smiles_col='smiles', name_col='name',
    n_clusters=8, n_diverse=30,
)
print(result_df[['name','mw','logp','qed','cluster','maxmin_selected']].head(8).to_string())

In [ ]:
# ── Final cheatsheet + file listing ─────────────────────────────────────────
import os

cheat = [
    'UMAP CHEMICAL SPACE ANALYSIS — COMPLETE REFERENCE',
    '',
    'PIP INSTALL',
    '  pip install rdkit umap-learn scikit-learn',
    '  pip install numpy pandas matplotlib seaborn plotly',
    '',
    'FINGERPRINT (industry standard = Morgan ECFP4)',
    '  fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)',
    '  arr = np.zeros((2048,)); DataStructs.ConvertToNumpyArray(fp, arr)',
    '',
    'UMAP SETTINGS',
    '  reducer = umap.UMAP(',
    '      n_components=2,',
    '      n_neighbors=15,    # 5=local  15=default  50=global',
    '      min_dist=0.1,      # 0.0=tight  0.1=default  0.5=spread',
    '      metric="jaccard",  # correct for binary fingerprints',
    '      random_state=42,   # reproducibility',
    '      n_jobs=-1,         # use all CPU cores',
    '  )',
    '  embedding = reducer.fit_transform(fp_matrix)  # (n_cpds, 2)',
    '  # New compounds (no refit): reducer.transform(new_fps)',
    '',
    'PLOTLY INTERACTIVE HTML',
    '  import plotly.express as px',
    '  fig = px.scatter(df, x="umap_x", y="umap_y", color="class",',
    '                   hover_name="name", hover_data={"mw","logp","smiles"})',
    '  fig.write_html("umap_map.html")  # standalone, no server needed',
    '',
    'MURCKO SCAFFOLD',
    '  from rdkit.Chem.Scaffolds import MurckoScaffold',
    '  core = MurckoScaffold.GetScaffoldForMol(mol)',
    '  scaffold_smi = Chem.MolToSmiles(core)',
    '',
    'MAXMIN DIVERSITY SELECTION',
    '  from rdkit.SimDivFilters import rdSimDivPickers',
    '  picker = rdSimDivPickers.MaxMinPicker()',
    '  idxs = picker.LazyBitVectorPick(fps_list, len(fps_list), n_pick)',
    '',
    'SCALING TIPS',
    '  >10k:  n_jobs=-1, low_memory=False (default)',
    '  >100k: umap(low_memory=True), batch fingerprinting',
    '  >500k: cuml.manifold.UMAP (GPU, RAPIDS)',
    '         or: fit on 50k subsample, transform() rest',
    '',
    'KEY PARAMETERS MEANING',
    '  n_neighbors=5:   very local, tight clusters',
    '  n_neighbors=50:  more global, smoother landscape',
    '  min_dist=0.0:    maximum cluster packing',
    '  min_dist=0.5:    spread out, good for global topology',
    '  metric=jaccard:  mathematically correct for bit FPs',
    '  metric=euclidean:faster, nearly equivalent in practice',
]
print('\n'.join(cheat))

print()
print('Files created:')
print('='*55)
files = [
    'umap_output/fingerprint_analysis.png',
    'umap_output/umap_parameters.png',
    'umap_output/umap_static.png',
    'umap_output/umap_interactive_2d.html',
    'umap_output/umap_logp.html',
    'umap_output/umap_interactive_3d.html',
    'umap_output/umap_properties.png',
    'umap_output/umap_scaffolds.png',
    'umap_output/diversity_selection.png',
    'umap_output/diverse_selection_maxmin.csv',
    'umap_output/compounds_umap.csv',
    'umap_output/umap_map.html',
]
for f in files:
    status = 'OK' if os.path.exists(f) else '--'
    print(f'  [{status}] {f}')